# Google Analytics Session Revenue Prediction
## DSC 288 Capstone — Milestone 3 Presentation Notes

---

**Team:** Justin Chanthabandith · Pooja Panchal · Jinxin Xiao  
**Course:** DSC 288R Capstone  
**Dataset:** Google Analytics Customer Revenue Prediction (Kaggle, 2018)

---

> This notebook contains the full written content for our Milestone 3 presentation.  
> Each section corresponds to a slide group and is written to be copied directly into PowerPoint slides.  
> No code is included — this is a narrative and content document only.


---
## Slide 1–2 · Background

### The Problem We Are Solving

Our project is about predicting **how much revenue a single website session will generate** using Google Analytics data from the Google Merchandise Store. More specifically, we are trying to answer two questions at once: first, will this session produce any revenue at all, and second, if it does, how much?

The target variable is `totals.transactionRevenue` — the dollar amount of a purchase made during a browsing session. If no purchase happened, the value is zero (or null in the raw data, which we treat as zero).

---

### Why Is This Problem Important?

If you have ever wondered how companies decide where to spend their advertising budget, this is the core of that decision. E-commerce companies run campaigns through paid search, organic search, social media, referral links, and direct traffic. All of these channels cost money. But the uncomfortable truth is that the overwhelming majority of website visits produce zero revenue — most people browse, look around, and leave without buying anything.

This creates a real and expensive problem. If a company has no way to distinguish between a session that is likely to convert and one that is just browsing, they end up spending the same amount of effort and money on both. That is wasteful in multiple ways:

- Ad budgets go toward users who will never buy, instead of being concentrated on high-intent visitors
- Marketing messages are generic because there is no prediction of who is worth personalizing for
- Revenue forecasts are unreliable because no one can estimate which sessions will actually generate money
- Customer support resources are spread thin instead of being prioritized toward high-value sessions

Accurate session revenue prediction gives a company the ability to fix all of these things. If you know that a particular session has a high probability of converting, you can show that user a different landing page, offer a discount, or prioritize them in a live chat queue. If you know a session is unlikely to buy, you stop spending money trying to push them toward a purchase.

Beyond the business case, this is also a genuinely interesting machine learning problem. The dataset is large, the target is almost entirely zeros with a heavily skewed tail for the positive cases, the features come in nested JSON format and need real preprocessing work, and building a model that actually learns something useful rather than just predicting zero for everything requires careful thinking at every step.


---
## Slide 3–4 · Literature Survey

### How Have People Tried to Solve This Before?

Customer revenue prediction and purchase likelihood modeling have been studied for a long time, and the approaches have changed significantly as the tools available to practitioners improved.

---

**Traditional and Rule-Based Approaches (pre-2010s)**

Before machine learning became widely accessible, businesses used simpler frameworks to understand customer behavior. The most common was the **RFM model** — Recency, Frequency, and Monetary value. The idea is straightforward: a customer who bought recently, buys often, and spends a lot is more valuable than one who bought once a long time ago. You score each customer on these three dimensions and segment them accordingly.

Alongside RFM, companies used **basic linear regression** for revenue prediction and **logistic regression** for binary classification of whether a session would convert. These methods are still used today because they are fast, interpretable, and easy to explain to non-technical stakeholders.

The problem with these approaches is that they assume the world is relatively linear and additive. In reality, whether someone buys something online depends on the combination of many factors interacting in ways that a linear model cannot easily capture. A user from paid search on a desktop in the United States during November behaves very differently from a user from social media on a mobile phone in India in July, even if some of their individual feature values look similar.

---

**Machine Learning Approaches (2010s–present)**

The 2018 Google Analytics Kaggle competition is the closest published benchmark to our exact problem, and the solutions that performed best were consistently **gradient boosted trees** — specifically XGBoost, LightGBM, and CatBoost. These models dominated the leaderboards because they handle nonlinear relationships naturally, work well with mixed numeric and categorical features, are robust to outliers and skewed distributions, and can capture the complex interaction effects that linear models miss.

Several patterns emerged consistently across top solutions:

- **Log-transforming the revenue target** was universal. Raw revenue is so skewed that models trained on it directly perform poorly. Log1p transformation brings the distribution into a range where regression models can learn meaningfully.
- **Two-step modeling** became a popular strategy. Instead of trying to predict revenue in one shot, top teams separated the problem: first predict whether any revenue will occur (classification), then predict how much revenue for the positive cases (regression). This handles the zero-inflation problem explicitly.
- **Visitor-level aggregate features** were consistently among the most predictive variables. A user who has purchased before is far more likely to purchase again. Features like prior session count, prior total revenue, and prior purchase rate gave models a strong cross-session signal.
- **User-level train/test splitting** was recognized as critical. If the same visitor appears in both training and test sets, the model can memorize visitor-level patterns and inflate its apparent performance. Proper evaluation requires splitting so that no visitor appears in both sets.

---

### Gaps Our Project Addresses

Most public implementations of this problem have at least one of the following issues that we specifically set out to fix:

| Problem in Prior Work | Our Solution |
|---|---|
| Visitor-level features computed with leakage (including current session in the aggregate) | We compute visitor history using only prior sessions, sorted by timestamp |
| Random train/test splits allowing the same visitor in both sets | We split exclusively by `fullVisitorId` using `GroupShuffleSplit` |
| Naive pandas loading that crashes or runs out of memory on 1.7M rows of nested JSON | We use Polars for all loading and preprocessing, converting to pandas only for the final sklearn step |
| No check on whether an EDA sample is representative of the full dataset | We explicitly compare our 50K sample to the full dataset before trusting any EDA conclusions |
| Single-model approach that ignores the zero-inflation structure | We compare a direct regression approach against a two-step classify-then-regress pipeline |


---
## Slide 5 · Why Machine Learning Can Help

### Is There a Strong Reason to Believe ML Is Better Here?

Yes, and we think it goes beyond the fact that this is a machine learning course. There are concrete structural reasons why ML — and specifically tree-based ML — is better suited to this problem than traditional statistical approaches.

---

**The relationships in this data are not linear**

The simplest way to see this is to think about what it would take for linear regression to model purchase behavior correctly. Linear regression assumes that each feature contributes an independent, additive amount to the prediction. But that is not how purchasing decisions work. Whether someone buys something online depends on the interaction of many factors simultaneously.

A high visit number might mean a loyal repeat customer who is likely to buy — or it could mean someone who visits often but never purchases. The same feature value leads to a completely different outcome depending on where the traffic came from, what device they used, and what their prior purchase history looks like. Linear regression cannot model this without you manually creating every interaction term, which is impractical at scale.

Random Forest and gradient boosting models handle this naturally. Every split in a decision tree is a conditional rule that already encodes an interaction. The model can learn "if the user came from Paid Search AND it is their third visit AND they are on desktop → higher purchase probability" without us ever writing that rule explicitly.

---

**The positive signal is extremely rare and needs to be found carefully**

Only about 1.3% of sessions in this dataset generate any revenue. A model that predicts zero for everything would be correct 98.7% of the time by raw accuracy, and it would have a very low MAE. But it would be completely useless for any business purpose.

This is a situation where naive approaches fail quietly. Traditional linear models trained on a zero-inflated target tend to push all their predictions toward zero because that minimizes their training loss. Tree-based models are better at partitioning the feature space in ways that isolate the rare high-revenue corner, especially when the target is log-transformed and the model is evaluated on RMSE rather than accuracy.

---

**Our expectation going in**

We expected tree-based models to clearly outperform both naive baselines and linear regression. We also expected the two-step model to outperform a single-stage regressor, because separating the classification step from the regression step seemed like the right way to handle the zero-inflation problem structurally.

As we will show in the results section, the first expectation was confirmed. The second was not — and understanding why turned out to be one of the more interesting findings of the project.


---
## Slide 6 · Dataset Details

### Where Does the Data Come From?

We are using the **Google Analytics Customer Revenue Prediction** dataset from Kaggle, published as part of a 2018 competition. The data comes from the **Google Merchandise Store** — Google's own e-commerce site where you can buy branded merchandise. It contains anonymized, session-level Google Analytics logs covering roughly 18 months of traffic from August 2016 through April 2018.

We did not generate this dataset. We used it as-is from the competition source, with all of our modifications being part of the feature engineering pipeline rather than any changes to the underlying raw data.

**Citation:** Google Analytics Customer Revenue Prediction, Kaggle (2018).  
https://www.kaggle.com/c/ga-customer-revenue-prediction

---

### Dataset Dimensions and Metadata

| Property | Value |
|---|---|
| Total sessions (rows) | ~1.7 million |
| Raw columns | 13 base columns |
| After JSON flattening | ~20 usable feature columns |
| Time range | August 2016 – April 2018 (~18 months) |
| Unique visitors | ~700,000+ |
| Sessions with positive revenue | ~1.3% of all sessions |
| Target variable | `totals.transactionRevenue` (in micros — we divide by 1,000,000 for USD) |
| Primary evaluation metric | RMSE on log1p-transformed revenue |

---

### How the Raw Data Is Structured

One of the first things that stands out about this dataset is that it is not a clean flat CSV. Several of the columns store their data as **nested JSON strings** inside each cell. For example, the `device` column does not just say "desktop" — it contains a full JSON object like:

```
{"browser": "Chrome", "operatingSystem": "Windows", "deviceCategory": "desktop", ...}
```

The columns that are stored this way are `device`, `geoNetwork`, `totals`, and `trafficSource`. To get anything useful out of them, you have to parse or extract the specific subfields you need. This is one of the reasons we use Polars instead of pandas — doing this kind of extraction on 1.7 million rows in pandas using `json_normalize` is slow and often crashes the kernel. Polars can do regex-based field extraction efficiently across all rows without loading the full nested structure.

---

### Key Dataset Challenges

**Zero inflation** is the single biggest challenge. About 98.7% of sessions have a transaction revenue of zero. This means the target variable is not a normal distribution that a regression model can learn from easily — it is a spike at zero with a very long, thin tail of positive values. Any model evaluated on raw accuracy would achieve ~98.7% by predicting zero for everything, which is completely useless.

**Revenue skew** adds another layer of difficulty. Among the ~1.3% of sessions that do generate revenue, the values are extremely right-skewed. A few sessions generate very large amounts while most positive sessions generate modest amounts. We address this by using log1p-transformed revenue as the actual regression target, which compresses the scale and makes the distribution more learnable.

**Nested JSON structure** means that standard CSV loading is not enough. Every row's device, network, traffic, and session information is buried inside strings that have to be parsed before they can be used as features.

**High-cardinality categoricals** are another practical problem. The `geoNetwork.country` field has 200+ unique values. `trafficSource.source` has thousands. Naive one-hot encoding of these columns would create an extremely sparse and wide feature matrix that is slow to work with and can cause overfitting on rare categories.

**Visitor leakage risk** is the most subtle challenge. The same visitor can appear in many sessions across the 18 months of data. If we compute aggregate features like "total revenue for this visitor" without checking temporal order, we can accidentally include future session data in a feature that is supposed to represent what we knew about the visitor at the time of the current session. This kind of leakage inflates model performance in training and makes real-world deployment unreliable.

**Structural missingness** in `totals.bounces` and `totals.newVisits` deserves a special mention. In the raw data, these fields are encoded as `"1"` when the condition is true and **null** (not `"0"`) when it is false. After numeric conversion and median imputation, these columns collapse to near-constant values and lose all their variance. Our correlation heatmap confirmed this — both columns showed zero correlation with everything. We dropped both from the feature set as a direct result of this EDA finding.


---
## Slide 7–9 · Feature Extraction and EDA

### The Data Cleaning Pipeline

Before we could do any analysis or modeling, we needed to transform the raw nested CSV into a clean, flat, analysis-ready table. Here is every step we took and the reasoning behind each one:

**Step 1 — Load with Polars instead of pandas**  
The full dataset is 1.7 million rows and several of the columns contain long JSON strings. Loading this with standard pandas and then calling `json_normalize` on every row is extremely slow and frequently crashes notebook kernels. We switched to Polars, which uses a more memory-efficient columnar format and lets us do regex-based field extraction without fully parsing the JSON structure on every row.

**Step 2 — Extract only the fields we need from the nested JSON columns**  
Instead of flattening every key from `device`, `geoNetwork`, `totals`, and `trafficSource`, we extract only the specific subfields our analysis uses. This keeps the working dataframe much smaller than if we had expanded every nested key.

**Step 3 — Drop the raw JSON columns immediately after extraction**  
As soon as we have the subfields we need, we drop the original nested columns. This frees memory right away and avoids carrying around large string columns through the rest of the pipeline.

**Step 4 — Parse and engineer the revenue target**  
`totals.transactionRevenue` is a string that needs to be converted to a float, and nulls need to be filled with zero (because null means no purchase, not that the value is unknown). From this we create three target columns: `revenue` (raw USD), `has_revenue` (binary 0/1 indicator), and `log_revenue` (log1p-transformed, which is the actual regression target).

**Step 5 — Parse timestamps into time-based features**  
The `date` field is stored as a YYYYMMDD string and `visitStartTime` is a Unix epoch integer. We convert both to datetime objects and extract year, month, day of week, and hour as usable features.

**Step 6 — Handle missing categoricals explicitly**  
Rather than dropping rows with null categorical values or imputing them with the most frequent value, we fill null categoricals with the literal string `"Missing"`. This turns missingness into its own category that the model can learn from, which matters here because missingness in web analytics data is often not random.

**Step 7 — Reduce high-cardinality categorical variables**  
For `trafficSource.source`, `device.browser`, and `geoNetwork.country` — which can have hundreds or thousands of unique values — we keep the top 25 most frequent values and map everything else to `"Other"`. This prevents the feature matrix from becoming enormous and noisy.

**Step 8 — One-hot encode all categoricals**  
We use sklearn's `OneHotEncoder(handle_unknown="ignore")` so that any categories that appear in the test set but not training are handled gracefully rather than throwing an error.

**Step 9 — Median imputation for numeric features**  
Missing numeric values are filled with the column median rather than the mean, because our numeric features are skewed and the median is more robust to outliers.

**Step 10 — Scale only for linear/logistic models**  
We apply `StandardScaler` to numeric features for Linear Regression and Logistic Regression pipelines, but not for Random Forest. Tree models split on rank order, not magnitude, so scaling does not affect them and would just add unnecessary computation.

**Step 11 — Split by visitor, not randomly**  
The final and most important step: we use `GroupShuffleSplit` with `fullVisitorId` as the group key for an 80/20 train/test split. This guarantees that no visitor appears in both training and test sets, preventing the model from memorizing visitor-level patterns and reporting inflated performance.

---

### Sample Representativeness Check

For most of the EDA exploration, we worked with a 50,000-row sample rather than the full 1.7 million rows, because iterating quickly on EDA is much easier at that scale. But before trusting any conclusions from the sample, we verified that it actually looked like the full dataset.

We compared the sample to the full data on five dimensions: positive revenue rate, channel grouping distribution, date range coverage, revenue percentile distribution, and unique visitor count. The sample matched well across all of these. The positive revenue rate was within a fraction of a percent of the full dataset, the channel mix was proportionally similar, and the date range covered the same 18-month period. This gave us confidence that conclusions drawn from the sample would generalize to the full data.

For modeling, we used the full dataset loaded through Polars to make sure our train/test split was operating over the complete visitor population.

---

### EDA Finding 1 — Zero Inflation and Revenue Skew

The most important structural finding from our EDA is how extreme the zero-inflation is. About 98.7% of sessions produce no revenue whatsoever. The remaining 1.3% generate all the revenue, and even within that group the distribution is strongly right-skewed — a small number of sessions generate very large amounts while most positive sessions are modest.

This finding directly shapes our entire modeling strategy. A regression model trained naively on raw revenue will push most predictions toward zero because that minimizes its loss on the overwhelming majority of sessions. Log-transforming the target compresses the scale of positive values and makes the regression task more learnable. And the zero-inflation is severe enough that we seriously considered whether separating the "will they buy?" question from the "how much will they spend?" question would give better results — which became our two-step model experiment.

---

### EDA Finding 2 — Session Engagement Features

We looked at how session engagement variables — specifically hits (total interactions in a session), pageviews, and visit number — compared between revenue-generating and non-revenue sessions.

The pattern was clear and consistent: sessions that generated revenue had notably higher mean hits, higher mean pageviews, and higher mean visit numbers than sessions that produced no revenue. This makes intuitive sense — users who are genuinely interested in buying tend to browse more pages, interact more with the site, and have visited before. These are not surprising findings, but they confirm that these features carry real predictive signal and justify keeping them in the feature set.

---

### EDA Finding 3 — Traffic Channel and Device

Not all traffic is equal. When we grouped sessions by their acquisition channel and computed the positive revenue rate for each group, Paid Search had the highest conversion rate at around 3.4%. This makes sense — users clicking on a paid search ad are actively searching for the product and showing purchase intent before they even arrive at the site. Direct traffic also converted well, which is consistent with returning customers who already know what they want.

Social media and Display advertising had very low conversion rates, around 0.3–0.4%. These channels bring in large volumes of traffic but most of those visitors are not in buying mode — they are browsing casually and are easy to distract. This is a well-documented pattern in e-commerce and our data confirmed it.

For device type, desktop sessions converted at roughly four times the rate of mobile sessions. This is also a widely observed pattern in online retail — the checkout experience on mobile is harder, and mobile users tend to browse more than they buy. Tablet was in between but much closer to mobile than desktop.

These findings justified including both channel grouping and device category as features. They carry real signal about purchase likelihood that the model should be able to exploit.

---

### EDA Finding 4 — Time-Based Patterns

We looked at positive revenue rates broken down by month, day of week, and hour of day. The monthly pattern showed a clear holiday spike in November and December, consistent with the Google Merchandise Store seeing higher purchase activity around the holiday shopping season. Summer months showed a slight dip. These seasonal effects are real and worth capturing as model features.

Day-of-week analysis showed that weekdays converted noticeably better than weekends. This is typical for a merchandise store — people tend to browse and buy during work hours more than on weekends, possibly because they are using work computers and are in a more transactional mindset.

Hourly patterns showed a peak in mid-morning to early afternoon, roughly 9am to 3pm, with lower conversion rates in early morning and late night hours. None of these time features are overwhelmingly predictive on their own, but they contribute moderate signal that tree-based models can incorporate as part of their decision logic.

---

### EDA Finding 5 — Visitor History

One of the strongest patterns we found was in repeat visitor behavior. When we aggregated sessions to the visitor level and looked at purchase rates by number of prior sessions, the pattern was dramatic: users on their first-ever visit had a very low purchase rate, while users who had visited 10 or more times before converted at roughly 7 times that rate.

This confirmed that visitor-level history features would be among the most predictive in the model. But it also raised the leakage problem we mentioned earlier: if we just compute total visitor revenue or purchase rate naively, we include the current session in the aggregate, which means the model could effectively see the answer during training. We addressed this by computing all visitor history features using only sessions that happened before the current one, sorted by visitor and timestamp.

---

### EDA Finding 6 — Dropping totals.bounces and totals.newVisits

Our correlation heatmap revealed something unexpected: `totals.bounces` and `totals.newVisits` showed zero or near-zero correlation with every other feature, including each other. This is not what we would expect if these were genuinely informative features.

The explanation is structural. In the raw Google Analytics data, `bounces` is encoded as `"1"` when a session bounced and **null** when it did not. It is not `"1"` vs `"0"` — the absence of a bounce is represented by the field being missing entirely. The same applies to `newVisits`. After Polars extraction converts these strings to numeric values, you get `1.0` or `NaN`. Median imputation then fills all the `NaN` values with a constant, collapsing the entire column to a near-constant value with no variance left.

Because all the variance was destroyed by imputation, these features cannot help a model learn anything. We dropped both from the feature set as a direct result of this finding. The signal that `newVisits` was supposed to carry — whether a visitor is new or returning — is already captured more reliably by `visitNumber` (first visits have `visitNumber = 1`).


---
## Slide 10–11 · Model Details

### What Models Did We Use and Why?

We evaluated five models total, moving from the most naive possible baselines up to our main machine learning approaches. The progression was intentional — each step adds a layer of complexity and the results tell us how much that complexity is actually helping.

---

### Baseline 1 — Zero Revenue Baseline

This model predicts `log_revenue = 0` for every single session, without looking at any features at all.

We included this because it represents the floor of what any useful model needs to beat. Given that 98.7% of sessions have zero revenue, this baseline is actually not terrible on MAE — it is correct most of the time in absolute terms. But it is completely useless for any business purpose because it never identifies a high-value session. More importantly, it performs poorly on RMSE, which penalizes large errors more heavily. When this model misses a high-revenue session, the squared error is large, and those large misses accumulate.

---

### Baseline 2 — Mean Log Revenue Baseline

This model predicts the training set mean of `log_revenue` for every session.

This tests whether any model is actually learning feature-specific patterns, or just predicting the overall average. If a model cannot beat mean prediction, it has not learned anything that generalizes. Beating this baseline is the minimum bar for any ML model to be considered useful.

---

### Model 3 — Linear Regression

A standard supervised learning model that fits a linear combination of all features to predict `log_revenue`.

For this model, numeric features are median-imputed and then scaled with `StandardScaler`. Categorical features are one-hot encoded after rare categories are grouped into `"Other"`. Both preprocessing and the model itself are wrapped in a single sklearn `Pipeline` so there is no risk of fit/transform leakage between train and test.

We included linear regression because it is the simplest real model. If the relationship between features and revenue were mostly linear and additive, linear regression would capture it well. The fact that it does not perform as well as Random Forest tells us something meaningful about the nonlinearity of the problem.

---

### Model 4 — Random Forest Regressor

Our main machine learning model. A Random Forest builds an ensemble of 100 decision trees, each trained on a different random bootstrap sample of the training data using a random subset of features, and averages their predictions.

**Configuration we used:**

| Parameter | Value | Reasoning |
|---|---|---|
| `n_estimators` | 100 | Enough trees for stable average predictions; increasing further has diminishing returns |
| `max_depth` | 12 | Limits how deep individual trees can grow, preventing them from memorizing the training data |
| `min_samples_leaf` | 10 | Each leaf node must have at least 10 samples, which smooths predictions and prevents overfitting on small groups |
| `random_state` | 42 | Ensures reproducibility |
| `n_jobs` | -1 | Uses all available CPU cores in parallel |

**Why Random Forest for this problem:**

Random Forest does not require the relationship between features and revenue to be linear. Each split in a decision tree is a conditional rule — the model can learn things like "if the user came from Paid Search AND has a visit number above 3 AND is on desktop → higher predicted revenue" without us ever specifying that interaction manually. This is exactly the kind of multi-way conditional reasoning that is needed to model purchase behavior well.

Random Forest is also naturally robust to the skewed distributions in our features. Because each tree splits on threshold values rather than raw magnitudes, extreme values in one feature do not dominate the way they might in a linear model. And because predictions are averaged across 100 trees, the variance from any single tree's idiosyncratic behavior is reduced.

For a problem with mixed numeric and categorical features, real nonlinearity, and sparse revenue signal, Random Forest is a well-justified choice as the primary ML baseline before moving to gradient boosting.

---

### Model 5 — Two-Step Logistic Regression + Random Forest

This model was designed explicitly to handle the zero-inflation problem by splitting revenue prediction into two separate tasks.

**Step 1 — Logistic Regression classifier:**  
We train a logistic regression model to predict `has_revenue` (0 or 1). We use `class_weight="balanced"` so that the model pays more attention to the rare positive class rather than just learning to predict zero for everything. This model outputs a `purchase_probability` for every test session.

**Step 2 — Random Forest regressor on positive sessions only:**  
We train a separate Random Forest model using only the training sessions where `has_revenue = 1`. This model learns to predict `log_revenue` given that a purchase is happening, without having to worry about all the zero cases. It outputs a `predicted_positive_revenue` for every test session.

**Combining the two predictions:**  
`final_prediction = purchase_probability × predicted_positive_revenue`

The intuition is that the final prediction should reflect both how likely a purchase is and how large it would be if it happened.

**Why this is theoretically attractive:**  
By separating the classification and regression tasks, each sub-model can specialize. The regressor is trained on a much cleaner, more normally distributed target (only positive revenue values, already log-transformed) without having to simultaneously figure out which sessions will produce anything at all. This feels like the right structural decomposition for a zero-inflated problem.

**Why it underperformed in practice:**  
This is the more interesting part of the story, and we cover it in detail in the results section.


---
## Slide 12–14 · Results and Observations

### Model Performance Table

| Model | RMSE (log revenue) ↓ | MAE (log revenue) ↓ |
|---|---:|---:|
| 🥇 **Random Forest Regressor** | **1.524** | **0.263** |
| Linear Regression | 1.732 | 0.414 |
| Mean Log Revenue Baseline | 1.828 | 0.379 |
| Zero Revenue Baseline | 1.838 | 0.190 |
| Two-Step Logistic + RF | 3.754 | 1.512 |

Lower is better for both metrics. The primary metric is RMSE on log-transformed revenue, which is the standard metric for this Kaggle competition.

---

### What the Results Tell Us

**Random Forest is the clear winner.** It achieved the best RMSE at 1.524, beating the zero baseline by about 17%. That is a meaningful improvement on a dataset this sparse, where naive approaches are hard to beat simply because most sessions truly have zero revenue. The result confirms that our feature set — session behavior, traffic source, device, geography, time features, and visitor history — contains real predictive signal that a tree-based model can learn from.

**Linear Regression also beat both baselines**, coming in at 1.732. This tells us that even a simple model can extract some signal from the features we engineered. The gap between Linear Regression and Random Forest (1.732 vs 1.524) reflects the nonlinear structure of the problem — the interactions between features that linear regression cannot model without manual feature crosses.

**The Zero Revenue Baseline had the lowest MAE (0.190) despite being the worst RMSE model.** This is a counterintuitive result that is worth understanding carefully. MAE measures the average absolute difference between predicted and actual values. Because 98.7% of sessions have zero revenue, always predicting zero is never far off in absolute terms for most sessions. The model is "close" to the right answer on almost all rows.

But RMSE squares the errors before averaging them. When the zero baseline misses a high-revenue session — say, a session that generated $500 worth of purchases — the error on that one session is enormous after squaring. Those large misses on rare high-value sessions drive the RMSE up significantly. This is exactly why RMSE is the right metric for this problem. The business value is in identifying the rare high-revenue sessions, not in being approximately correct on the zero sessions. A model that never identifies a high-revenue session is useless no matter how good its MAE looks.

**The Two-Step model's poor performance (RMSE 3.754) is the most interesting result.** It performed worse than all other models including both naive baselines, which was not what we expected given that the approach is theoretically well-suited to zero-inflated targets.

After thinking about what went wrong, we believe the core issue is **logistic regression probability miscalibration on a severely imbalanced target**. With only 1.3% of sessions being positive, logistic regression with `class_weight="balanced"` is pushed to aggressively flag potential positive sessions. The class weighting causes it to inflate predicted probabilities upward to compensate for the class imbalance. When those inflated probabilities are multiplied by the Random Forest's revenue predictions, the result is a large predicted revenue for many sessions that should be zero. This introduces systematic overestimation that compounds throughout the test set.

In other words, the two-step model did not fail because the idea was wrong — it failed because the first step produced unreliable probabilities, and those unreliable probabilities poisoned the final prediction through multiplication. A better classifier (gradient boosting for the classification step), probability calibration (Platt scaling or isotonic regression), or a different way of combining the two predictions might fix this.

---

### What We Infer from These Results

The overall picture that emerges from our modeling results is fairly clear:

Tree-based models are well-suited to this problem and linear approaches are not sufficient. The nonlinear feature interactions that drive purchase behavior — channel × device × visitor history × timing — are exactly the kind of patterns that Random Forest and gradient boosting are designed to capture.

The two-step architecture is the right conceptual framing for a zero-inflated regression problem, but implementation details matter enormously. A miscalibrated first step does not just add some error — it multiplies error into the final prediction. Getting the two-step model to work well requires either a much stronger classifier or a fundamentally different way of combining the two outputs.

The most natural next step is testing gradient boosted models (XGBoost or LightGBM) as a direct replacement for Random Forest in the single-stage pipeline. Based on published Kaggle solutions for this exact dataset, those models should improve RMSE further while keeping the same general architecture we have already built.


---
## Slide 15 · Further Items Before Final Submission

### What We Plan to Complete in the Next Two Weeks

**Test XGBoost and LightGBM as the primary models**  
Both gradient boosting frameworks consistently outperform vanilla Random Forest on structured tabular e-commerce data. They use a different training strategy — building trees sequentially where each tree corrects the residuals of the previous ones — which tends to produce better predictive performance than Random Forest's parallel ensemble approach. LightGBM is particularly fast on large datasets and should run efficiently with our full 1.7M row setup. This is our highest-priority next step.

**Tune the Random Forest hyperparameters more carefully**  
Our current configuration uses sensible defaults but was not optimized. Running a random search over `n_estimators`, `max_depth`, and `min_samples_leaf` may squeeze additional improvement out of the model before we move to gradient boosting.

**Add feature importance analysis**  
Both Random Forest and gradient boosting models can output feature importances, which tell us which features contribute most to the predictions. This is useful for two reasons: it helps us explain the model to stakeholders in plain language, and it may reveal features that are contributing noise rather than signal, allowing us to simplify the feature set.

**Improve and properly evaluate the two-step model**  
We want to try a gradient boosting classifier for the first step instead of logistic regression, and apply probability calibration to bring the output probabilities closer to true likelihoods. We also want to report AUC-ROC and a precision-recall curve for the classification step, so we can evaluate the purchase prediction component separately from the overall RMSE.

**Add result visualizations**  
A predicted versus actual scatter plot will help us see whether the model is systematically biased in any direction — for example, if it consistently underestimates very high revenue sessions or overestimates low-revenue ones. Residual plots will also help identify any patterns in the errors that suggest additional features or transformations.

**Polish the final notebook narrative**  
The analysis notebook needs some final cleanup to make sure every section tells a coherent story from data loading through results, with all dead-end or debugging cells removed.

---

### Risks and How We Are Handling Them

**Class imbalance causing poor classifier performance**  
Risk level: High. With 1.3% positive rate, any classifier is at risk of effectively ignoring the minority class. We are addressing this with `class_weight="balanced"` and will also evaluate with AUC-ROC and precision-recall AUC rather than accuracy, which is misleading on imbalanced data.

**Visitor leakage inflating model performance**  
Risk level: Medium. This is already addressed — our visitor history features use only prior sessions and our train/test split is visitor-level. But we will double-check this in the final validation step.

**Overfitting to the training visitor population**  
Risk level: Medium. The Google Merchandise Store visitor population may have changed over the 18-month data collection period. Grouped cross-validation (rather than a single split) would give a more robust estimate of generalization performance.

**High-cardinality categoricals causing noisy encoding**  
Risk level: Medium. We are already grouping rare categories into "Other", but the choice of top_k=25 was somewhat arbitrary. Testing different cutoffs may improve performance slightly.

**Kernel memory and compute constraints**  
Risk level: Low. We are already using Polars for all preprocessing and only converting to pandas at the final sklearn step. This keeps memory usage manageable even on the full 1.7M row dataset.

**Two-step model remaining uncompetitive**  
Risk level: High for the current implementation. We believe a gradient boosting classifier with calibrated probabilities should improve this significantly, but if not, we will acknowledge the two-step approach as a negative result with a clear explanation of why it failed.


---
## Slide 16 · References

1. **Kaggle.** *Google Analytics Customer Revenue Prediction Competition*, 2018.  
   https://www.kaggle.com/c/ga-customer-revenue-prediction  
   *(Primary data source and competition benchmark)*

2. **Breiman, L.** (2001). *Random Forests.* Machine Learning, 45(1), 5–32.  
   *(Foundational paper for the Random Forest algorithm used as our primary model)*

3. **Chen, T., and Guestrin, C.** (2016). *XGBoost: A Scalable Tree Boosting System.*  
   Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining.  
   *(Framework we plan to test in the final submission; consistently top-performing on tabular data)*

4. **Ke, G. et al.** (2017). *LightGBM: A Highly Efficient Gradient Boosting Decision Tree.*  
   Advances in Neural Information Processing Systems (NeurIPS).  
   *(Second gradient boosting framework we plan to evaluate; known for speed on large datasets)*

5. **Scikit-learn Developers.** *Scikit-learn: Machine Learning in Python.*  
   https://scikit-learn.org  
   *(Used for all modeling pipelines: Pipeline, GroupShuffleSplit, RandomForestRegressor, LogisticRegression, ColumnTransformer)*

6. **Polars Developers.** *Polars User Guide.*  
   https://pola.rs  
   *(Used for all data loading and preprocessing on the full 1.7M row dataset)*

7. **RFM Model — Recency, Frequency, Monetary Value.**  
   Hughes, A.M. (1994). *Strategic Database Marketing.* Probus Publishing.  
   *(Traditional customer segmentation approach discussed in the literature survey)*
